# Results for model: gemma3:12b

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

def solve():
    train = pl.read_parquet("./jane-street-real-time-market-data-forecasting/train.parquet")

    def top_quantile(series: pl.Series, quantile: float = 0.85) -> pl.Series:
        """Calculates the top quantile of a series."""
        return series.quantile(quantile)

    def feature_engineering(df: pl.DataFrame, date_id_col: str = "date_id", feature_col: str = "feature_00", responder_col: str = "responder_6", window_size: int = 10) -> pl.DataFrame:
        """Calculates the global top quantile of a feature relative to a responder across rolling batches."""
        df = df.with_columns(
            pl.col(date_id_col).rolling_window(window_size).mean().alias("date_id_mean")
        )
        df = df.with_columns(
            pl.col(feature_col).rolling_window(window_size).quantile(0.85).alias("feature_00_top_quantile")
        )
        return df

    train = feature_engineering(train)

    X = train[["date_id_mean", "feature_00_top_quantile"]]
    y = train["responder_6"]

    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
    model.fit(X, y)

    return model

if __name__ == "__main__":
    model = solve()
    print("Model trained successfully.")